# 00.04 — Derivatives & Gradients

**You should already know:** vectors and dot products (00.01), array shapes and matrix notation (00.02), polynomial expansion, and differentiation of a polynomial.
**By the end, you will be able to:**

1. state that a one-variable derivative measures the local rate of change of an output as its input changes, and interpret the sign of that rate;
2. compute the derivative of the quadratic $q(t)=\frac{1}{2}(t-5)^2$ and evaluate it at $t=3$;
3. identify a partial derivative as the rate of change with respect to one input while the other input is held fixed;
4. construct and interpret the gradient as the ordered vector of partial derivatives of a two-weight loss;
5. calculate the prediction, residual, squared-error loss, two partial derivatives, and gradient for the frozen example by hand;
6. implement that exact calculation in NumPy with explicit shape checks and deterministic assertions; and
7. explain, without performing an optimization algorithm, why later ML lessons can use gradient information to choose changes to parameters.

**Difficulty:** ★★☆☆☆ · **Time:** about 40–50 minutes

## 1. The Problem

A dot-product prediction is below its target. How can we describe, one weight at a time, how the selected loss changes nearby?

With one input variable, the answer is a derivative — the local rate of change at a single point. With two weights, each weight gets its own rate, and the ordered pair of rates is a gradient.

This lesson computes both for one visible, exact, two-feature example. The calculation does not claim that the example is trained, accurate, representative, causal, or a real data set.

## 2. The Toy Example

All values below are **dimensionless teaching quantities**. Mathematical vectors are displayed as column vectors; NumPy code uses rank-1 arrays of shape `(2,)`.

$$
\mathbf{x}=\begin{bmatrix}1\\2\end{bmatrix},\qquad
\mathbf{w}=\begin{bmatrix}w_1\\w_2\end{bmatrix}
=\begin{bmatrix}1\\1\end{bmatrix},\qquad
y=5.
$$

The scalar prediction is the dot product:

$$\hat y = \mathbf{w}^{\mathsf T}\mathbf{x} = w_1 x_1 + w_2 x_2.$$

The signed residual and half squared-error loss are:

$$r = \hat y - y,\qquad L(\mathbf{w}) = \tfrac{1}{2}\,r^2.$$

The $\tfrac{1}{2}$ makes the derivative values transparent; it has no special empirical meaning. $L$ is a dimensionless squared-error quantity — not a percentage, probability, accuracy, or proof of useful predictions.

## 3. Intuition Before Math

Before working with two weights, start with one input. Consider the one-variable loss:

$$q(t) = \tfrac{1}{2}(t-5)^2.$$

At $t=3$, the loss value is $q(3) = \tfrac{1}{2}(-2)^2 = 2$. But how fast is $q$ changing near $t=3$?

A **derivative** answers this: it is the local rate of change at a single point, not the average slope over a large interval. Graphically, it is the slope of the tangent line touching the curve at that point.

The derivative turns out to be $q'(3)=-2$. The negative sign means that near $t=3$, a small increase in $t$ is associated with a local decrease in this loss.

Note: $q(3)=2$ (the loss value) and $q'(3)=-2$ (the local rate) are different quantities. A positive loss and a negative derivative are not contradictory — the function is positive and locally decreasing at the same point.

## 4. Mathematical Derivation

### One-variable derivative

For a function $q$, its **derivative** at $t$, when the limit exists, is

$$q'(t) = \lim_{h\to 0}\frac{q(t+h)-q(t)}{h}.$$

Apply this to $q(t)=\tfrac{1}{2}(t-5)^2$:

$$
\begin{aligned}
\frac{q(t+h)-q(t)}{h}
&= \frac{(t+h-5)^2-(t-5)^2}{2h}\\
&= \frac{2h(t-5)+h^2}{2h}\\
&= (t-5)+\frac{h}{2} \qquad (h\ne0).
\end{aligned}
$$

Taking the limit gives $q'(t)=t-5$, so $q'(3)=3-5=-2$.

### Two-weight loss

Writing the loss by its two coordinates:

$$L(w_1,w_2) = \tfrac{1}{2}(w_1+2w_2-5)^2.$$

A **partial derivative** changes one named weight while holding the other fixed. Expand the squared expression first:

$$L(w_1,w_2) = \tfrac{1}{2}w_1^2 + 2w_2^2 + 2w_1 w_2 - 5w_1 - 10w_2 + \tfrac{25}{2}.$$

Differentiate with respect to $w_1$ (treating $w_2$ as a constant):

$$\frac{\partial L}{\partial w_1} = w_1 + 2w_2 - 5.$$

Differentiate with respect to $w_2$ (treating $w_1$ as a constant):

$$\frac{\partial L}{\partial w_2} = 2w_1 + 4w_2 - 10.$$

### The gradient

The **gradient** preserves the order of the weight vector $\mathbf{w}=[w_1,w_2]$:

$$
\nabla_{\mathbf{w}}L
= \begin{bmatrix}\partial L/\partial w_1 \\ \partial L/\partial w_2\end{bmatrix}
= \begin{bmatrix}w_1+2w_2-5 \\ 2w_1+4w_2-10\end{bmatrix}.
$$

The gradient has shape `(2,)` because there is one component for each entry of $\mathbf{w}$.

## 5. Manual Walkthrough

| Symbol | Meaning | Frozen value |
|---|---|---:|
| $\mathbf{x}$ | feature vector | $[1, 2]$ |
| $\mathbf{w}$ | weight vector | $[1, 1]$ |
| $y$ | target scalar | $5$ |
| $\hat y = \mathbf{w}^{\mathsf T}\mathbf{x}$ | prediction | $1{\cdot}1+1{\cdot}2=3$ |
| $r = \hat y - y$ | residual | $3-5=-2$ |
| $L = \tfrac{1}{2}r^2$ | loss | $\tfrac{1}{2}(-2)^2=2$ |
| $\partial L/\partial w_1$ | rate for $w_1$ at $(1,1)$ | $1+2(1)-5=-2$ |
| $\partial L/\partial w_2$ | rate for $w_2$ at $(1,1)$ | $2(1)+4(1)-10=-4$ |
| $\nabla_{\mathbf{w}}L$ | ordered gradient | $[-2, -4]$ |

Both gradient components are negative at this point. This means that increasing either weight a small amount is associated with a local reduction in this selected toy loss. This is local information — it does not prove that the point is optimal, that a model is useful, or that either feature causes the target.

### Independent algebra check

The component formulas factor as:

$$
\frac{\partial L}{\partial w_1} = w_1+2w_2-5 = r, \qquad
\frac{\partial L}{\partial w_2} = 2(w_1+2w_2-5) = 2r.
$$

Reassembling:

$$
\nabla_{\mathbf{w}}L
= r\begin{bmatrix}1\\2\end{bmatrix}
= (\mathbf{w}^{\mathsf T}\mathbf{x}-y)\,\mathbf{x}
= (-2)\begin{bmatrix}1\\2\end{bmatrix}
= \begin{bmatrix}-2\\-4\end{bmatrix}.
$$

This matches both partial-derivative calculations. It is an algebraic consistency check, not a new differentiation method.

## 6. Teaching Implementation

The code below mirrors the manual calculation with pure Python/NumPy. All vectors are 1-D float arrays of shape `(2,)`, and all scalar quantities have shape `()`. Before calculating, the code checks that inputs have the required shapes.

In [ ]:
import numpy as np

# ── frozen inputs ────────────────────────────────────────────
x = np.array([1.0, 2.0])
w = np.array([1.0, 1.0])
y = np.float64(5.0)

assert x.shape == (2,), f"x shape: {x.shape}"
assert w.shape == (2,), f"w shape: {w.shape}"
assert np.ndim(y) == 0, f"y ndim: {np.ndim(y)}"

# ── forward pass ─────────────────────────────────────────────
y_hat = w @ x               # prediction (scalar)
r = y_hat - y                # residual  (scalar)
L = 0.5 * r * r              # loss      (scalar)

print(f"ŷ  = {float(y_hat)}")
print(f"r  = {float(r)}")
print(f"L  = {float(L)}")

# ── one-variable warm-up ─────────────────────────────────────
t = np.float64(3.0)
q_val = 0.5 * (t - 5.0) ** 2   # q(3)
qp_val = t - 5.0                # q'(3)

print(f"q(3)  = {float(q_val)}")
print(f"q'(3) = {float(qp_val)}")

# ── gradient from expanded polynomial ────────────────────────
def partial_w1_expanded(w1, w2):
    """∂L/∂w₁ = w₁ + 2w₂ − 5"""
    return w1 + 2.0 * w2 - 5.0

def partial_w2_expanded(w1, w2):
    """∂L/∂w₂ = 2w₁ + 4w₂ − 10"""
    return 2.0 * w1 + 4.0 * w2 - 10.0

dL_dw1 = partial_w1_expanded(w[0], w[1])
dL_dw2 = partial_w2_expanded(w[0], w[1])
gradient_expanded = np.array([dL_dw1, dL_dw2])

print(f"∂L/∂w₁ = {float(dL_dw1)}")
print(f"∂L/∂w₂ = {float(dL_dw2)}")
print(f"∇L (expanded) = {gradient_expanded.tolist()}")

## 7. Verification of Fixed Teaching Calculations

The assertions below verify every fixed result from the manual walkthrough. They use `np.testing.assert_allclose` for floating-point values and explicit shape checks.

In [ ]:
# Shapes
assert x.shape == (2,)
assert w.shape == (2,)
assert np.ndim(y) == 0
assert np.ndim(y_hat) == 0
assert np.ndim(r) == 0
assert np.ndim(L) == 0
assert gradient_expanded.shape == (2,)

# Forward pass values
np.testing.assert_allclose(y_hat, 3.0, err_msg="prediction mismatch")
np.testing.assert_allclose(r, -2.0, err_msg="residual mismatch")
np.testing.assert_allclose(L, 2.0, err_msg="loss mismatch")

# One-variable warm-up
np.testing.assert_allclose(q_val, 2.0, err_msg="q(3) mismatch")
np.testing.assert_allclose(qp_val, -2.0, err_msg="q'(3) mismatch")

# Gradient components
np.testing.assert_allclose(dL_dw1, -2.0, err_msg="∂L/∂w₁ mismatch")
np.testing.assert_allclose(dL_dw2, -4.0, err_msg="∂L/∂w₂ mismatch")
np.testing.assert_allclose(
    gradient_expanded, [-2.0, -4.0], err_msg="gradient mismatch"
)

print("PASS")

## 8. Independent Cross-Check

The vector formula $(\mathbf{w}^{\mathsf T}\mathbf{x}-y)\,\mathbf{x}$ is computed independently and compared against the two component formulas from the expanded polynomial.

In [ ]:
# Vector-formula gradient (independent expression)
gradient_vector = (w @ x - y) * x

assert gradient_vector.shape == (2,)
np.testing.assert_allclose(
    gradient_vector, [-2.0, -4.0], err_msg="vector gradient mismatch"
)

# Both paths agree
np.testing.assert_allclose(
    gradient_expanded, gradient_vector,
    err_msg="expanded and vector gradients disagree",
)

print(f"∇L (vector)   = {gradient_vector.tolist()}")
print(f"∇L (expanded) = {gradient_expanded.tolist()}")
print("PASS: both gradient paths agree")

## 9. Visual Explanation

![Two side-by-side plots. The left plot shows a U-shaped curve of q(t) = one-half times (t minus 5) squared over t from 0 to 6. A red dashed tangent line touches the curve at the marked point (3, 2) with slope negative 2. The right plot shows diagonal contour lines of L(w1, w2) over w1 and w2 from negative 1 to 4 at levels 0.0, 0.5, 1.0, 2.0, 4.0, 8.0, and 12.5. A red dot marks (1, 1) labelled L equals 2. A black arrow from (1, 1) points toward approximately (0.6, 0.2), labelled 0.2 times gradient L and local increase direction.](assets/derivatives_gradients.png)

**A derivative describes the local rate of change; a gradient collects one such rate per weight.** In the left panel, the tangent line at $t=3$ has slope $q'(3)=-2$, showing the loss is locally decreasing. In the right panel, labelled contours show $L(w_1,w_2)$, and the black arrow at $(1,1)$ points in the direction of local increase of the loss, scaled by $0.2$ for readability. The arrow is not a descent step, convergence path, or model-quality indicator.

## 10. Failure Modes & Pitfalls

- **"The derivative is the average slope between two points."** No: $q'(3)=-2$ is the local rate at $t=3$, defined by a limit as $h \to 0$, not a secant slope.
- **"A loss of 2 means 2% error."** No: $L=2$ is a dimensionless squared-error value, not a percentage, probability, or accuracy.
- **"A negative derivative means the loss is negative."** No: $q(3)=2$ (positive value) and $q'(3)=-2$ (negative rate) are different quantities.
- **"A partial derivative changes both weights."** No: $\partial L / \partial w_1$ changes $w_1$ while holding $w_2$ fixed.
- **"Gradient entries can be reordered."** No: the gradient matches $\mathbf{w}=[w_1,w_2]$. Swapping gives $[-4,-2]$, which misassigns rates to weights.
- **"Computing a gradient proves the model is optimal or useful."** No: a gradient is local information at one point. It proves no optimum, no generalization, and no causality.

### Spot the bug

A learner computes the two partial derivatives correctly ($-2$ and $-4$) but writes:

$$\nabla L = [-4, -2]$$

Which component is misplaced? Use the partial-derivative formulas and the weight-vector order to identify and correct the error.

<details><summary>Answer</summary>

The weight vector is $\mathbf{w}=[w_1,w_2]$, so the gradient must be $[\partial L/\partial w_1, \partial L/\partial w_2] = [-2, -4]$. The learner swapped the two entries.
</details>

## 11. Exercises

### 1. Loss versus derivative
At $t=3$: $q(3)=2$ and $q'(3)=-2$. (a) What does $q(3)=2$ measure? (b) What does $q'(3)=-2$ measure? (c) Why are a positive loss and a negative derivative not contradictory?

<details><summary>Answer</summary>

(a) The value of the loss at $t=3$, a dimensionless squared-error quantity. (b) The local rate of change: near $t=3$, a small increase in $t$ locally decreases the loss. (c) They are different quantities; a function can be positive and decreasing at the same point.
</details>

### 2. Gradient coordinate order
Given $\mathbf{w}=[w_1,w_2]$, $\partial L/\partial w_1=-2$, $\partial L/\partial w_2=-4$, write the gradient. What goes wrong if you write $[-4,-2]$?

<details><summary>Answer</summary>

$\nabla L = [-2,-4]$. Writing $[-4,-2]$ assigns the $w_2$-rate to the $w_1$-position and vice versa.
</details>

### 3. Spot the bug
A learner presents $\nabla L=[-4,-2]$. Using $\partial L/\partial w_1 = w_1+2w_2-5$ and $\partial L/\partial w_2=2w_1+4w_2-10$, evaluate at $(1,1)$ and correct the gradient.

<details><summary>Answer</summary>

$\partial L/\partial w_1 = 1+2-5=-2$ (first entry); $\partial L/\partial w_2 = 2+4-10=-4$ (second entry). The entries are swapped. Corrected: $\nabla L=[-2,-4]$.
</details>

### 4. Shape check
Before code runs, state the shapes of $\mathbf{x}$, $\mathbf{w}$, $\hat y$, $L$, and $\nabla L$.

<details><summary>Answer</summary>

$\mathbf{x}$: `(2,)`, $\mathbf{w}$: `(2,)`, $\hat y$: `()` (scalar), $L$: `()` (scalar), $\nabla L$: `(2,)` — one entry per weight.
</details>

Gradient-descent updates, the chain rule, directional derivatives, and finite-difference methods are topics for later lessons.

## 12. References and Further Reading

1. Strang, G. and Herman, E. J. *Calculus Volume 1*, §3.1 "Defining the Derivative." OpenStax, 2016. [Open-access textbook](https://openstax.org/books/calculus-volume-1/pages/3-1-defining-the-derivative).
2. Auroux, D. *18.02SC Multivariable Calculus*, MIT OpenCourseWare, Fall 2010. [Course page](https://ocw.mit.edu/courses/18-02sc-multivariable-calculus-fall-2010/pages/syllabus/).
3. Goodfellow, I., Bengio, Y., and Courville, A. *Deep Learning*, ch. 4, §4.3 "Gradient-Based Optimization." MIT Press, 2016. [Online chapter](https://www.deeplearningbook.org/contents/numerical.html).
4. NumPy Developers. `numpy.dot` API reference, NumPy 1.26. [API page](https://numpy.org/doc/1.26/reference/generated/numpy.dot.html).